# Task 1.2 — Key Assumptions
**Paper:** *Training SVMs Without Offset* — Steinwart, Hush & Scovel, JMLR 2011

**Student:** Kush Agarwal | **Roll No.:** 230050 | NST, Rishihood University, Sonipat

## Assumption 1

**Assumption:** The training data must be approximately centered at the origin in the feature space — that is, after any preprocessing, the class means should be close to zero.

**Why the method needs it:** Since b = 0, the decision hyperplane is geometrically fixed to pass through the origin. In a standard SVM, the bias b acts as a free parameter that shifts the boundary to wherever the data needs it. Without b, the only way the boundary can be positioned correctly is if the data is already roughly symmetric around the origin. The paper explicitly recommends input normalization (zero-mean, unit-variance) before training the offset-free SVM for exactly this reason.

**Violation scenario:** A medical imaging dataset where pixel intensity values all lie in [0, 255] and no normalization is applied. Both class clusters (say, tumour vs. no-tumour patches) would live in the strictly positive orthant, far from the origin. The optimal hyperplane would need a large negative bias b to separate them — something the offset-free formulation simply cannot provide.

**Paper reference:** Section 1 (Introduction), where the authors discuss conditions under which removing b is appropriate and explicitly recommend feature normalization before applying the method.

---

## Assumption 2

**Assumption:** The kernel function k must be a valid Mercer kernel — the kernel matrix K (where Kᵢⱼ = k(xᵢ, xⱼ)) must be positive semi-definite (PSD) for any finite training set.

**Why the method needs it:** The dual objective W(α) = Σαᵢ − (1/2)αᵀQα with Qᵢⱼ = yᵢyⱼKᵢⱼ is a concave quadratic maximization problem — but only when Q is PSD, which holds if and only if K is PSD (Mercer's condition). Concavity has two important consequences: it guarantees a unique global maximum, and it guarantees that coordinate ascent converges to the global optimum. If K is not PSD, the problem becomes non-concave, which means local maxima exist and the 1D coordinate updates are no longer guaranteed to make progress toward the global solution.

**Violation scenario:** A custom similarity function defined as k(x, x') = x·x' − 1 (dot product minus a constant). Subtracting a constant can produce a kernel matrix with negative eigenvalues, violating PSD. On such a problem, the coordinate ascent loop may oscillate or converge to a suboptimal point, and the convergence theorem of the paper (Theorem 1) would not apply.

**Paper reference:** Section 2 (dual derivation, Equation 2) where the PSD structure is implicitly required; Theorem 1 (Section 5) where the finite convergence proof relies on the dual being a concave quadratic.

---

## Assumption 3

**Assumption:** The regularization parameter C must be finite and reasonably chosen — the algorithm's finite convergence guarantee depends on C being bounded.

**Why the method needs it:** Theorem 1 in Section 5 gives an explicit upper bound on the number of iterations before the algorithm terminates. That bound is a function of C — roughly O(C²/ε²) where ε is the stopping tolerance. If C is very large, the feasible box [0, C]ⁿ becomes very wide, and dual variables can take large values, meaning the algorithm may need a very large number of steps to converge. In the limit C → ∞ (hard-margin SVM), the convergence guarantee breaks down entirely, particularly if the data is not perfectly separable.

**Violation scenario:** A dataset with mislabelled examples combined with an extremely large C (e.g., C = 10⁶). The solver will try to assign very large α values to the noisy points (forcing them correctly classified at great cost), which requires many more iterations than normal and produces a badly overfitted model that generalizes poorly.

**Paper reference:** Section 2 (the clip operation to [0, C] in Procedure 1, Eq. 3) and Section 5 (Theorem 1, where the iteration bound is written explicitly as a function of C).